In [9]:
import sys
from pathlib import Path

# Set root path agar modul app dapat diimport dengan benar
ROOT = Path.cwd().resolve()
while ROOT.name != "SIGAP-TANI-BACKEND" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root added to sys.path: {ROOT}")

Project root added to sys.path: D:\Projek\2026-07\SIGAP-TANI-BACKEND


### 1. Inisialisasi & Pengambilan Data Histori OPT

In [10]:
import pandas as pd

from app.src.services.opt_service import OptService
from app.config.db import SessionLocal


db = SessionLocal()

optService = OptService(db)

data = optService.get_histori_serangan()

df = pd.DataFrame(data)

df

,id,bulan,tahun,kecamatan_id,opt_id,jumlah_serangan,musim_tanaman,luas_puso,created_at,updated_at
0,6387,12,2025,1,2,1.0,25/26,0.0,2026-07-25 12:41:36,2026-07-25 12:41:36
1,6388,12,2025,1,3,1.0,25/26,0.0,2026-07-25 12:41:36,2026-07-25 12:41:36
2,6389,12,2025,2,1,1.0,25/26,0.0,2026-07-25 12:41:36,2026-07-25 12:41:36
3,6390,12,2025,2,2,1.5,25/26,0.0,2026-07-25 12:41:36,2026-07-25 12:41:36
4,6391,12,2025,2,3,1.0,25/26,0.0,2026-07-25 12:41:36,2026-07-25 12:41:36
...,...,...,...,...,...,...,...,...,...,...
6438,54,1,2016,21,1,1.6,15/16,0.0,2026-07-25 12:41:10,2026-07-25 12:41:10
6439,55,1,2016,21,3,3.0,15/16,0.0,2026-07-25 12:41:10,2026-07-25 12:41:10
6440,56,1,2016,21,5,1.3,15/16,0.0,2026-07-25 12:41:10,2026-07-25 12:41:10
6441,57,1,2016,22,1,0.7,15/16,0.0,2026-07-25 12:41:10,2026-07-25 12:41:10


In [11]:
# Validasi kelengkapan data per tahun, kecamatan, dan opt (12 bulan)
df_check = df.copy()

# normalisasi tipe data
df_check["tahun"] = df_check["tahun"].astype(int)
df_check["bulan"] = df_check["bulan"].astype(int)
df_check["kecamatan_id"] = df_check["kecamatan_id"].astype(int)
df_check["opt_id"] = df_check["opt_id"].astype(int)

# aggregate agar tiap kombinasi unik tidak double-count
df_agg = (
    df_check.groupby(
        ["tahun", "kecamatan_id", "opt_id", "bulan"],
        as_index=False,
        dropna=False
    )["jumlah_serangan"]
    .sum()
)

all_years = sorted(df_agg["tahun"].dropna().unique().tolist())
all_kec = sorted(df_agg["kecamatan_id"].dropna().unique().tolist())
all_opt = sorted(df_agg["opt_id"].dropna().unique().tolist())
all_months = list(range(1, 13))

# buat kartesian product lengkap: tahun x kecamatan x opt x bulan
full_index = pd.MultiIndex.from_product(
    [all_years, all_kec, all_opt, all_months],
    names=["tahun", "kecamatan_id", "opt_id", "bulan"]
).to_frame(index=False)

# gabung dengan data yang ada
df_full_valid = full_index.merge(
    df_agg,
    on=["tahun", "kecamatan_id", "opt_id", "bulan"],
    how="left"
)

# isi 0 jika tidak ada data untuk bulan tertentu
df_full_valid["jumlah_serangan"] = df_full_valid["jumlah_serangan"].fillna(0.0)

# status kelengkapan tiap kombinasi (tahun, kecamatan, opt)
summary = (
    df_full_valid.groupby(["tahun", "kecamatan_id", "opt_id"], as_index=False)
    .agg(
        bulan_tersedia=("bulan", "nunique"),
        total_serangan=("jumlah_serangan", "sum"),
        lengkap_12_bulan=("bulan", lambda s: s.nunique() == 12),
    )
)

# lihat kombinasi yang belum lengkap
missing = summary[~summary["lengkap_12_bulan"]].copy()
missing["bulan_kosong"] = [
    [m for m in all_months if m not in df_full_valid[
        (df_full_valid["tahun"] == row["tahun"]) &
        (df_full_valid["kecamatan_id"] == row["kecamatan_id"]) &
        (df_full_valid["opt_id"] == row["opt_id"])
    ]["bulan"].tolist()]
    for _, row in missing[["tahun", "kecamatan_id", "opt_id"]].iterrows()
]

summary = summary.sort_values(["tahun", "kecamatan_id", "opt_id"]).reset_index(drop=True)
missing = missing.sort_values(["tahun", "kecamatan_id", "opt_id"]).reset_index(drop=True)

summary
missing

,tahun,kecamatan_id,opt_id,bulan_tersedia,total_serangan,lengkap_12_bulan,bulan_kosong


In [12]:
# Pastikan setiap kombinasi (tahun, musim_tanaman, kecamatan_id, opt_id, bulan) punya baris
# Jika tidak ada, isi jumlah_serangan = 0

df_clean = df.copy()

# normalisasi tipe data
df_clean["bulan"] = df_clean["bulan"].astype(int)
df_clean["kecamatan_id"] = df_clean["kecamatan_id"].astype(int)
df_clean["opt_id"] = df_clean["opt_id"].astype(int)

# kalau ada duplikasi, gabungkan jumlah_serangan-nya
df_agg = (
    df_clean.groupby(
        ["tahun", "musim_tanaman", "kecamatan_id", "opt_id", "bulan"],
        as_index=False,
        dropna=False
    )["jumlah_serangan"]
    .sum()
)

all_years = sorted(df_agg["tahun"].dropna().unique().tolist())
all_musim = sorted(df_agg["musim_tanaman"].dropna().unique().tolist())
all_kec = sorted(df_agg["kecamatan_id"].unique().tolist())
all_opt = sorted(df_agg["opt_id"].unique().tolist())
all_months = list(range(1, 13))

# buat kombinasi lengkap
full_index = pd.MultiIndex.from_product(
    [all_years, all_musim, all_kec, all_opt, all_months],
    names=["tahun", "musim_tanaman", "kecamatan_id", "opt_id", "bulan"]
).to_frame(index=False)

# gabung dengan data asli
df_full = full_index.merge(
    df_agg,
    on=["tahun", "musim_tanaman", "kecamatan_id", "opt_id", "bulan"],
    how="left"
)

# isi 0 untuk bulan yang tidak ada
df_full["jumlah_serangan"] = df_full["jumlah_serangan"].fillna(0.0)

# urutkan agar rapi
df = df_full.sort_values(["tahun", "musim_tanaman", "kecamatan_id", "opt_id", "bulan"]).reset_index(drop=True)

df

,tahun,musim_tanaman,kecamatan_id,opt_id,bulan,jumlah_serangan
0,2016,15/16,1,1,1,0.00
1,2016,15/16,1,1,2,0.75
2,2016,15/16,1,1,3,0.75
3,2016,15/16,1,1,4,0.00
4,2016,15/16,1,1,5,0.00
...,...,...,...,...,...,...
332635,2025,25/26,22,6,8,0.00
332636,2025,25/26,22,6,9,0.00
332637,2025,25/26,22,6,10,0.00
332638,2025,25/26,22,6,11,0.00
